# Week 3
First some exploration to see if we can shrink the parquet without losing `user_id` signal

In [41]:
import duckdb
from datetime import datetime

PARQUET_PATH = "../2022_place_canvas_history_uid.parquet"
# PARQUET_PATH = "../test.parquet"
START_TIME = datetime(int(2022), int(4), int(1), int(12))
END_TIME = datetime(int(2022), int(4), int(1), int(18))

In [ ]:
COLOR_NAME_MAPPING = {
    "#E4ABFF"   : "Mauve",
    "#FFA800"   : "Web Orange",
    "#D4D7D9"   : "Iron",
    "#811E9F"   : "Seance",
    "#FF3881"   : "Wild Strawberry",
    "#FFF8B8"   : "Buttermilk",
    "#00756F"   : "Pine Green",
    "#BE0039"   : "Monza",
    "#94B3FF"   : "Anakiwa",
    
}

In [11]:
query = f"""
    SELECT
        DISTINCT pixel_color
    FROM
        '{PARQUET_PATH}'
"""

duckdb.query(query)

┌─────────────┐
│ pixel_color │
│   varchar   │
├─────────────┤
│ #E4ABFF     │
│ #FFA800     │
│ #D4D7D9     │
│ #811E9F     │
│ #FF3881     │
│ #FFF8B8     │
│ #00756F     │
│ #BE0039     │
│ #94B3FF     │
│ #493AC1     │
│    ·        │
│    ·        │
│    ·        │
│ #6A5CFF     │
│ #009EAA     │
│ #00A368     │
│ #898D90     │
│ #FFB470     │
│ #6D482F     │
│ #3690EA     │
│ #FFD635     │
│ #FF99AA     │
│ #B44AC0     │
├─────────────┤
│   32 rows   │
│ (20 shown)  │
└─────────────┘

In [4]:
query = f"""
    SELECT
        timestamp,
        coordinate
    FROM
        '{PARQUET_PATH}'
    WHERE
        LENGTH(coordinate) > 9
"""

duckdb.query(query)

┌─────────────────────┬─────────────────────┐
│      timestamp      │     coordinate      │
│      timestamp      │       varchar       │
├─────────────────────┼─────────────────────┤
│ 2022-04-04 01:22:50 │ 1349,1718,1424,1752 │
│ 2022-04-03 23:29:52 │ 297,1750,364,1813   │
│ 2022-04-03 23:05:04 │ 298,1805,329,1839   │
│ 2022-04-03 23:08:50 │ 257,1736,296,1780   │
│ 2022-04-03 23:03:29 │ 298,1770,334,1803   │
│ 2022-04-03 23:10:36 │ 251,1805,296,1812   │
│ 2022-04-03 23:12:51 │ 271,1835,296,1859   │
│ 2022-04-04 04:12:20 │ 23,1523,172,1792    │
│ 2022-04-04 04:29:41 │ 51,1691,154,1807    │
│ 2022-04-04 04:16:10 │ 44,1652,165,1899    │
│ 2022-04-04 19:29:49 │ 1372,1472,1406,1497 │
│ 2022-04-04 19:29:49 │ 1373,1400,1419,1436 │
│ 2022-04-04 19:29:49 │ 1371,1438,1418,1472 │
│ 2022-04-04 19:29:50 │ 1375,1355,1424,1399 │
│ 2022-04-04 15:58:33 │ 551,1311,562,1342   │
│ 2022-04-04 15:59:10 │ 547,1330,550,1342   │
│ 2022-04-01 14:44:08 │ 862,540,868,544     │
│ 2022-04-01 14:46:23 │ 862,540,87

# Rank Colors By Distinct Users
Report a ranking of colors by the number of distinct users who placed those during the specified timeframe.

In [ ]:
query = f"""
    SELECT
        pixel_color,
        COUNT(DISTINCT user_id) AS user_count
    FROM
        '{PARQUET_PATH}'
    WHERE
        timestamp >= '{START_TIME}' AND
        timestamp <= '{END_TIME}'
    GROUP BY
        pixel_color
    ORDER BY
        user_count DESC
"""

print(duckdb.query(query))

┌─────────────┬────────────┐
│ pixel_color │ user_count │
│   varchar   │   int64    │
├─────────────┼────────────┤
│ #000000     │     412124 │
│ #FF4500     │     299003 │
│ #FFFFFF     │     270212 │
│ #2450A4     │     244148 │
│ #FFD635     │     201025 │
│ #51E9F4     │     117115 │
│ #FF99AA     │     116719 │
│ #7EED56     │     102042 │
│ #00A368     │      95611 │
│ #811E9F     │      77678 │
│ #FFA800     │      77006 │
│ #3690EA     │      69604 │
│ #D4D7D9     │      38707 │
│ #898D90     │      37131 │
│ #9C6926     │      30551 │
│ #B44AC0     │      29539 │
├─────────────┴────────────┤
│ 16 rows        2 columns │
└──────────────────────────┘

# Calculate Average Session Length
A session is in progress until a 15-minute window of inactivity is reached. A user must have placed more than one pixel to have had a session.

Return the average session length in seconds during the specified timeframe.

In [ ]:
query = f"""
    WITH
        time_diff AS (
            SELECT
                user_id,
                
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp >= '{START_TIME}' AND
                timestamp <= '{END_TIME}'
        )
"""

duckdb.query(query)

┌──────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────┐
│                                         user_id                                          │      timestamp      │
│                                         varchar                                          │      timestamp      │
├──────────────────────────────────────────────────────────────────────────────────────────┼─────────────────────┤
│ WYFuP/nwVCIBrw5XOVYKsHyo/fJpOJcIXCm75iKLqEz92BATvW5OWI6lginRvFoFfuWOYGiRrN9UQ6wUx9tAAQ== │ 2022-04-01 15:38:01 │
│ 1Fie0j8msAiBmD5+NfV4SdY6ilMDHV9XJc6zWdhgAyrc8H2ZPcM5fqcDoxFENial6ZwtzGIAInbhBbz9kzTZPQ== │ 2022-04-01 15:38:01 │
│ qvqMCBzdQyIL1ET+iZvXilEjCrt7cjq3oPG3uUad8tWi1XSnvEsusSpnfWT3YGPaOa+MaCoPmz3vTomjmf5tjA== │ 2022-04-01 15:38:01 │
│ TGLWFhne7tzE8iBhUWm7K3m6SrH+2xBl599XMhdtvV6RqV7nry5gu7Ja/Xt/HRe1PHCqsN0wX+o3pWTIJIz0+Q== │ 2022-04-01 15:38:01 │
│ C1E2rtkIcBP+omujs+YcSHfdWfCBpcxq6uIQpSlekwfyIqZFOJ6uEWAfB629KpVWih7Lg0G5QoPku9

# Pixel Placement Percentiles
Calculate the 50th, 75th, 90th, and 99th percentiles of the number of pixels placed by users during the specified timeframe.

In [42]:
query = f"""
    WITH
        dist AS (
            SELECT
                user_id,
                COUNT(*) AS freq
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp >= '{START_TIME}' AND
                timestamp <= '{END_TIME}'
            GROUP BY
                user_id
        )
    SELECT
        quantile_cont(freq, 0.50 ORDER BY freq ASC) AS '50_percentile',
        quantile_cont(freq, 0.75 ORDER BY freq ASC) AS '75_percentile',
        quantile_cont(freq, 0.90 ORDER BY freq ASC) AS '90_percentile',
        quantile_cont(freq, 0.99 ORDER BY freq ASC) AS '99_percentile'
    FROM
        dist
"""
duckdb.query(query)

┌───────────────┬───────────────┬───────────────┬───────────────┐
│ 50_percentile │ 75_percentile │ 90_percentile │ 99_percentile │
│    double     │    double     │    double     │    double     │
├───────────────┼───────────────┼───────────────┼───────────────┤
│           2.0 │           4.0 │          11.0 │          33.0 │
└───────────────┴───────────────┴───────────────┴───────────────┘

# Count First-Time Users
Count how many users placed their first pixel ever within the specified timeframe.

In [35]:
START_TIME = datetime(int(2022), int(4), int(1), int(12))
END_TIME = datetime(int(2022), int(4), int(1), int(18))

query = f"""
    WITH
        during_timeframe AS (
            SELECT
                DISTINCT user_id
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp BETWEEN '{START_TIME}' AND '{END_TIME}'
        ),
        before_timeframe AS (
            SELECT
                DISTINCT user_id
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp < '{START_TIME}'
        )
    SELECT
        count(*) AS new_users
    FROM
        before_timeframe AS btf
    RIGHT JOIN during_timeframe AS dtf ON
        btf.user_id = dtf.user_id
    WHERE
        btf.user_id IS NULL
"""
duckdb.query(query)

┌───────────┐
│ new_users │
│   int64   │
├───────────┤
│   1062702 │
└───────────┘

In [39]:
import random
bigrams = [["hello", "world"], ["world", "is"], ["is", "lame"]]
firstword = "hello"

test = [tup for tup in bigrams if tup[0] == firstword]
print(random.choice([b[1] for b in bigrams if b[0] == firstword.lower()]))

world


In [ ]:
padding = (3 * width % 4) % 4

rowsize = 3 *width + padding